# On-device continual triage with **online SDFT** (LFM2.5-230M) — 3-way, 3 regimes

A phone-hosted 230M model learns your **drifting** attention policy — triage each
notification as **INTERRUPT** / **LATER** / **ARCHIVE** — from **implicit feedback**
(open now / let it wait / never opened), **no gold labels**, one `batch_size=1` LoRA
step at a time. The learned policy lives in a **~2.8 MB adapter**, so serving is a
bare prompt: no growing in-context cheat-sheet (ICL), no retrieval index (RAG).

**Two drifts, three regimes across a realistic week** — 50% regular **weekdays**,
20% **on-call**, 30% **off-hours** (evenings + the weekend), streamed
`weekday → on-call → off-hours`. Several categories flip on each drift — monitoring
alerts `ARCHIVE → INTERRUPT` when you go on-call; manager pings
`INTERRUPT → LATER → ARCHIVE` across all three; social
`ARCHIVE → ARCHIVE → INTERRUPT` once you're off the clock. The weights track whichever
regime you're living in: on our seeded MPS run each regime peaks *inside its own
window* (weekday 1.00, on-call 1.00 — on just 20% of the stream — off-hours 0.75).

**3-way is genuinely harder than binary** on a 230M — `LATER` and `ARCHIVE` both mean
"don't buzz me", so they partly conflate. Two tricks matter: every `batch_size=1`
update replays one item from *each* other class, and because 3-way can *over-train
and decay* past its peak we keep the best adapter on the current policy — the same
**probe guardrail / auto-rollback** the repo's `sdft.online` package ships. We also
sweep the context size k for ICL and RAG and compare against each baseline's *best* k.

**Standalone** — no `git clone`; the one input it needs, the seeded dataset the
repo's scripts trained on, is fetched straight from the repo at start. Paper:
[Shenfeld et al., 2026](https://self-distillation.github.io/SDFT)
([arXiv](https://arxiv.org/abs/2601.19897)). Base:
[Liquid AI LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M).


## Setup
Colab: **Runtime → GPU** (T4 is plenty). Installs deps and drops stale `torchao`.

In [ ]:
%pip install -q "transformers>=4.54" "trl>=0.19" "peft>=0.15" "datasets>=2.19" "accelerate>=0.33" matplotlib
%pip uninstall -y -q torchao
print("setup ok — 3-way / 3-regime online SDFT triage demo")

## Fetch the seeded dataset (the same one the repo's scripts trained on)

One `urllib` call replaces ~100 lines of generator code: the repo ships
[`data/inbox_triage.json`](https://github.com/lin826/Local-SDFT/blob/main/data/inbox_triage.json)
— the 60-item drifting stream, the three 12-item held-out eval sets, and the
config, every item with its prompt pre-rendered. The labels encode this latent
policy (bold = a drift flip):

| Category        | weekday   | on-call       | off-hours     |
| --------------- | --------- | ------------- | ------------- |
| `mgr_project`   | INTERRUPT | **LATER**     | **ARCHIVE**   |
| `calendar_soon` | INTERRUPT | INTERRUPT     | **LATER**     |
| `teammate_fyi`  | LATER     | **ARCHIVE**   | ARCHIVE       |
| `monitoring`    | ARCHIVE   | **INTERRUPT** | **ARCHIVE**   |
| `promo`         | ARCHIVE   | ARCHIVE       | **LATER**     |
| `social`        | ARCHIVE   | ARCHIVE       | **INTERRUPT** |
| `receipt`       | LATER     | **ARCHIVE**   | **LATER**     |

Blocks are class-balanced under their own policy (10/10/10, 4/4/4, 6/6/6) across
a realistic 50/20/30 week. The vocabulary trap survives the move to data:
`receipt` ("your **payment** succeeded") and `monitoring` ("**payment** latency")
share words but land on different rungs — you must learn the *policy*, not the
keywords. (The generator lives in `scripts/triage_common.py` in the repo.)

In [ ]:
import copy, json, random, re, urllib.request
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "LiquidAI/LFM2.5-230M"
DATA_URL = "https://raw.githubusercontent.com/lin826/Local-SDFT/main/data/inbox_triage.json"

with urllib.request.urlopen(DATA_URL) as fh:
    data = json.load(fh)
cfg = data["config"]
SEED = cfg["seed"]
ACTIONS = tuple(cfg["actions"])         # INTERRUPT / LATER / ARCHIVE
REGIMES = tuple(cfg["regimes"])         # weekday / on-call / off-hours
STREAM_LEN, EVAL_N = cfg["stream_len"], cfg["eval_n"]
DRIFTS = tuple(cfg["drifts"])           # weekday items 1-30 | on-call 31-42 | off-hours 43-60

CHECKPOINTS = (6, 12, 18, 24, 30, 36, 42, 48, 54, 60)
LORA_R, LORA_ALPHA = 16, 32
LORA_TARGET = r".*self_attn\.(q|k|v|out)_proj"
LR, MAX_NEW = 1e-3, 8
K_SWEEP = (3, 6, 9, 12)                 # context sizes tried for BOTH ICL and RAG
TEACHER_SHOTS, REPLAY, STEPS_PER_ITEM = 2, 16, 5   # 3-way wants a touch more gradient/item

stream = data["stream"]
evals = {int(k): v for k, v in data["evals"].items()}
eval_cur = evals[3]   # the current policy is the final regime = off-hours
print(f"fetched {len(stream)} stream + {sum(map(len, evals.values()))} eval items; regimes {REGIMES}")

## Model + pipeline helpers (inlined)

In [ ]:
def device():
    if torch.cuda.is_available(): return "cuda"
    if getattr(torch.backends,"mps",None) and torch.backends.mps.is_available(): return "mps"
    return "cpu"
DEV = device(); print("device:", DEV)

def load_tok():
    t = AutoTokenizer.from_pretrained(MODEL_NAME)
    if t.pad_token is None: t.pad_token = t.eos_token
    t.padding_side = "left"; return t
def load_base():
    dt = torch.float16 if DEV=="cuda" else torch.float32
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dt)
    return m if DEV=="cuda" else m.to(DEV)
def to_dev(enc, model):
    d = next(model.parameters()).device
    return {k:(v.to(d) if torch.is_tensor(v) else v) for k,v in enc.items()}

def demo_msg(it, a): return [{"role":"user","content":it["prompt"]},{"role":"assistant","content":a}]
def build_msgs(it, demos=None):
    m=[]
    for d,a in demos or []: m += demo_msg(d,a)
    m.append({"role":"user","content":it["prompt"]}); return m

ACTION_RE = re.compile(r"\b(INTERRUPT|LATER|ARCHIVE)\b", re.I)
def parse_action(t):
    m = ACTION_RE.search(t or ""); return m.group(1).upper() if m else "NONE"
def prompt_tokens(tok, msgs):
    txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return len(tok(txt, add_special_tokens=False)["input_ids"])
def accuracy(items, gens):
    return sum(parse_action(g)==it["action"] for it,g in zip(items,gens))/max(len(items),1)

@torch.inference_mode()
def generate(model, tok, items, demos_fn, bs=8, max_new=MAX_NEW):
    model.eval(); outs=[]
    for s in range(0,len(items),bs):
        batch=items[s:s+bs]
        txts=[tok.apply_chat_template(demos_fn(it),tokenize=False,add_generation_prompt=True) for it in batch]
        enc=to_dev(tok(txts,return_tensors="pt",padding=True,add_special_tokens=False),model)
        out=model.generate(**enc,max_new_tokens=max_new,do_sample=False,pad_token_id=tok.pad_token_id)
        for t in tok.batch_decode(out[:,enc["input_ids"].shape[1]:],skip_special_tokens=True):
            outs.append(t.strip())
    return outs

def retriever(store):
    toks=[set(re.findall(r"\w+",s["prompt"].lower())) for s in store]
    def r(it,k):
        q=set(re.findall(r"\w+",it["prompt"].lower()))
        order=sorted(range(len(store)),key=lambda i:-len(q&toks[i]))
        return [(store[i],store[i]["action"]) for i in order[:k]]
    return r

# persistent-optimizer online updater (Adam momentum across the stream) + grad clipping
def make_updater(model, tok):
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=LR)
    eos = tok.eos_token or ""
    dev = next(model.parameters()).device
    def step(rows, steps):
        model.train(); model.config.use_cache = False
        for k in range(steps):
            r = rows[k % len(rows)]
            ptxt = tok.apply_chat_template([{"role":"user","content":r["prompt"]}],
                                           tokenize=False, add_generation_prompt=True)
            ids_p = tok(ptxt, add_special_tokens=False)["input_ids"]
            ids_c = tok(r["target"] + eos, add_special_tokens=False)["input_ids"]
            ids = torch.tensor([ids_p + ids_c], device=dev)
            labels = torch.tensor([[-100]*len(ids_p) + ids_c], device=dev)  # completion-only loss
            model(input_ids=ids, labels=labels).loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)                  # keep bs=1 steps stable
            opt.step(); opt.zero_grad()
        model.config.use_cache = True; model.eval()
    return step
print("helpers ready")

## A look at the fetched stream

In [ ]:
from collections import Counter
for p in (1,2,3):
    print(f"phase {p} ({REGIMES[p-1]}):", dict(Counter(it["action"] for it in stream if it["phase"]==p)))
def flip(cat):   # how one category's label moves across the three regimes
    return [next(it["action"] for it in stream if it["phase"]==p and it["category"]==cat) for p in (1,2,3)]
print("\ntriple-flip — SAME manager ping, three regimes:", flip("mgr_project"))
print("social flips ARCHIVE->ARCHIVE->INTERRUPT:", flip("social"))
print("\nsample item:\n", stream[0]["prompt"][:230])

## Baselines with a k sweep: how much context do ICL and RAG *really* need?

ICL's cheat-sheet is k current-policy demos (round-robin over the three actions,
hand-kept fresh); RAG retrieves k nearest neighbours from the full 60-item decision
store. We try every k and keep each method's **best** (ties go to the smaller,
cheaper k). Watch RAG: its store is dominated by weekday decisions, so small k
retrieves *stale* answers — and past its sweet spot, more retrieval brings back
more staleness.

In [ ]:
tok = load_tok(); base = load_base()
retrieve = retriever(stream)
phase_cur = [it for it in stream if it["phase"]==3]
per_action = {a: [it for it in phase_cur if it["action"]==a] for a in ACTIONS}
def icl_demos_k(k):   # round-robin so the cheat-sheet stays balanced across actions
    picks = [per_action[ACTIONS[s % 3]][s // 3] for s in range(k)]
    return [(it, it["action"]) for it in picks]
def demos_for(method, it, k): return icl_demos_k(k) if method=="ICL" else retrieve(it,k)

sweep = {}
for method in ("ICL", "RAG"):
    for k in K_SWEEP:
        acc = accuracy(eval_cur, generate(base,tok,eval_cur,lambda it: build_msgs(it, demos_for(method,it,k)),bs=8))
        tks = sum(prompt_tokens(tok,build_msgs(it,demos_for(method,it,k))) for it in eval_cur)/EVAL_N
        sweep[(method,k)] = dict(acc=acc, tok=tks)
        print(f"  {method} k={k:2d}: acc={acc:.2f}  tok/query={tks:.0f}")
icl_k = max(K_SWEEP, key=lambda k: (sweep[("ICL",k)]["acc"], -k))
rag_k = max(K_SWEEP, key=lambda k: (sweep[("RAG",k)]["acc"], -k))
print(f"\nbest: ICL k={icl_k}, RAG k={rag_k}")

arms = {}
arms["ZS"] = dict(acc=accuracy(eval_cur, generate(base,tok,eval_cur,lambda it: build_msgs(it),bs=8)),
                  tok=sum(prompt_tokens(tok,build_msgs(it)) for it in eval_cur)/EVAL_N)
arms[f"ICL k={icl_k}"] = dict(acc=sweep[("ICL",icl_k)]["acc"], tok=sweep[("ICL",icl_k)]["tok"])
arms[f"RAG k={rag_k}"] = dict(acc=sweep[("RAG",rag_k)]["acc"], tok=sweep[("RAG",rag_k)]["tok"])
for k,v in arms.items(): print(f"  {k:10s} acc={v['acc']:.2f} tok/q={v['tok']:.0f}")

## The online SDFT loop: self-distill targets → `batch_size=1` stream → guardrail

The model makes its **own** call on every incoming item (with a couple of recent
decisions in context); your observed behaviour reinforces it when right, corrects it
when wrong — the target is always a bare action, never a hand-written gold answer.
`torch.manual_seed` makes the LoRA init (and so the whole run) repeatable. At each
checkpoint we probe all three regime policies; during the final regime the guardrail
snapshots the best off-hours adapter and rolls back to it before serving.

In [ ]:
# self-distill targets: the model's OWN guess, supervised only by the observed action
def guess_demos(i): return [(stream[j],stream[j]["action"]) for j in range(max(0,i-TEACHER_SHOTS),i)]
guesses = generate(base, tok, list(range(len(stream))), lambda i: build_msgs(stream[i], guess_demos(i)), bs=8)
rows = [{"prompt":it["prompt"],"target":it["action"]} for it in stream]
print("reinforced (model already right):", sum(parse_action(g)==it["action"] for it,g in zip(stream,guesses)), "/", len(rows))

torch.manual_seed(SEED)   # LoRA init + dropout masks — makes the run repeatable
model = get_peft_model(base, LoraConfig(r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=0.05,
                                        target_modules=LORA_TARGET,task_type="CAUSAL_LM"))
update = make_updater(model, tok)
curve = {"pos":[], "weekday":[], "oncall":[], "offhours":[]}
best = {"acc":-1.0, "pos":None, "state":None}
buf=[]; rr=random.Random(SEED)
for i,row in enumerate(rows):
    buf=(buf+[row])[-REPLAY:]
    batch=[row]   # pair the fresh item with one replayed item from EACH other class
    for a in ACTIONS:
        pool=[b for b in buf[:-1] if b["target"]==a]
        if a!=row["target"] and pool: batch.append(rr.sample(pool,1)[0])
    update(batch, STEPS_PER_ITEM)
    if (i+1) in CHECKPOINTS:
        a = {p: accuracy(evals[p], generate(model,tok,evals[p],lambda it:build_msgs(it),bs=8)) for p in (1,2,3)}
        curve["pos"].append(i+1); curve["weekday"].append(a[1]); curve["oncall"].append(a[2]); curve["offhours"].append(a[3])
        if (i+1) > DRIFTS[1] and a[3] >= best["acc"]:      # off-hours window: snapshot best-on-current
            best = {"acc":a[3], "pos":i+1, "state":copy.deepcopy(get_peft_model_state_dict(model))}
        print(f"  streamed {i+1:2d}: weekday={a[1]:.2f} on-call={a[2]:.2f} off-hours={a[3]:.2f}")
if best["state"] is not None:                              # auto-rollback to the best off-hours adapter
    set_peft_model_state_dict(model, best["state"])
    print(f"  probe guardrail: served best off-hours adapter (streamed {best['pos']}, acc {best['acc']:.2f})")
arms["Online-SDFT"] = dict(acc=best["acc"] if best["state"] else curve["offhours"][-1], tok=arms["ZS"]["tok"])
print("\nfinal (off-hours policy):", {k:round(v["acc"],2) for k,v in arms.items()})

## The figure: best-of-sweep baselines vs one bare-prompt adapter, across three regimes

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory
colors={"ZS":"#9aa0a6","ICL":"#e8710a","RAG":"#d93025","Online-SDFT":"#1a73e8"}
regime_colors={"weekday":"#7b3fa0","on-call":"#1a73e8","off-hours":"#0b8043"}
def col(n):
    for k,c in colors.items():
        if n.startswith(k): return c
    return "#5f6368"
zs_tok = arms["ZS"]["tok"]
fig,(axA,axB)=plt.subplots(1,2,figsize=(12.6,4.9))
for n,d in arms.items():
    axA.scatter(d["tok"],d["acc"]*100,s=170,color=col(n),zorder=3,edgecolor="white",linewidth=1.5)
    axA.annotate(n,(d["tok"],d["acc"]*100),textcoords="offset points",
                 xytext=(8, -5 if n.startswith("RAG") else 4),fontsize=10.5,fontweight="bold",color=col(n))
axA.set_xlabel("Recurring prompt tokens / query  (paid on every notification)")
axA.set_ylabel("Accuracy on current policy (%)"); axA.set_ylim(0,105); axA.grid(alpha=.25)
axA.axvspan(0,zs_tok+22,color="#1a73e8",alpha=.05)
axA.text((zs_tok+22)/2,6,"bare-prompt zone\n(weights carry the policy)",ha="center",fontsize=8.5,color="#1a73e8",style="italic")
axA.set_title("A.  Best-of-sweep baselines vs a bare prompt",fontweight="bold")

ref_arm=max((arms[f"ICL k={icl_k}"],arms[f"RAG k={rag_k}"]),key=lambda a:a["acc"])
ref=ref_arm["acc"]*100   # the winning frozen baseline, priced at ITS tokens
mult=max(1,round(ref_arm["tok"]/max(zs_tok,1)))
axB.axhline(ref,color="#e8710a",ls=":",lw=1.6)
axB.text(1, ref-2.5 if ref>93 else ref+2, f"best ICL / RAG on off-hours  (+{mult}× tokens every call)",
         fontsize=7.6, color="#e8710a", ha="left", va="top" if ref>93 else "bottom")
for x in DRIFTS: axB.axvline(x,color="#5f6368",ls="--",lw=1.2)
# no legend box: the curves share colors with the phase sub-titles below the axis
for key,regime in [("weekday","weekday"),("oncall","on-call"),("offhours","off-hours")]:
    axB.plot(curve["pos"],[v*100 for v in curve[key]],"-o",color=regime_colors[regime],lw=2.2,ms=4.5)
if best["pos"]:
    axB.scatter([best["pos"]],[best["acc"]*100],marker="*",s=300,color=regime_colors["off-hours"],
                edgecolor="white",linewidth=1.2,zorder=5)
    axB.annotate("probe keeps\nthis adapter",(best["pos"],best["acc"]*100),textcoords="offset points",
                 xytext=(0,-24 if best["acc"]>0.88 else 10),fontsize=7.5,
                 color=regime_colors["off-hours"],ha="center",fontweight="bold")
phase_axis = blended_transform_factory(axB.transData, axB.transAxes)
bounds = [0, *DRIFTS, STREAM_LEN]
for start,end,regime in zip(bounds,bounds[1:],REGIMES):
    axB.axvspan(start,end,color=regime_colors[regime],alpha=.045,zorder=0)
    axB.text((start+end)/2,-0.115,f"{regime}\nitems {start+1}–{end}",transform=phase_axis,
             ha="center",va="top",fontsize=8.3,color=regime_colors[regime],fontweight="bold")
axB.set_xlim(0,STREAM_LEN+2)
axB.set_xlabel("Items streamed (one batch_size=1 update each)",labelpad=36)
axB.set_ylabel("Held-out accuracy (%)")
axB.set_ylim(0,105); axB.grid(alpha=.25)
axB.set_title("B.  Weights track each regime — two drifts, one adapter",fontweight="bold")
fig.suptitle("On-device 3-way triage across 3 regimes · LFM2.5-230M · policy in a ~2.8 MB LoRA adapter, no gold labels",
             fontweight="bold",y=1.02)
fig.tight_layout(); plt.show()

## One item, four minds
An off-hours **social** push — off the clock it flips to **INTERRUPT** ("your
friends' posts DO deserve a buzz on Saturday"), where the base model files everything
as `LATER`. We show the first such item the served adapter gets right.

In [ ]:
socials = [it for it in eval_cur if it["category"]=="social"]
def act(adapter_off, demos, it):
    if adapter_off:
        with model.disable_adapter():
            return parse_action(generate(model,tok,[it],lambda x:build_msgs(x,demos),bs=1)[0])
    return parse_action(generate(model,tok,[it],lambda x:build_msgs(x,demos),bs=1)[0])
# pick a social item the served adapter gets right but the frozen base doesn't
q = next((it for it in socials if act(False,None,it)==it["action"] and act(True,None,it)!=it["action"]), socials[0])
print("PROMPT:\n", q["prompt"], "\n\ncorrect (off-hours):", q["action"], "\n")
extra = arms[f"ICL k={icl_k}"]["tok"] - arms["ZS"]["tok"]
print("  ZS          ->", act(True, None, q))
print(f"  ICL k={icl_k}     ->", act(True, icl_demos_k(icl_k), q), f"  (+{extra:.0f} tokens every call)")
print(f"  RAG k={rag_k}     ->", act(True, retrieve(q,rag_k), q))
print("  Online-SDFT ->", act(False, None, q), "  (bare prompt)")

## What to try next
1. **Change the policy (or make regimes longer)** — the generator lives in [scripts/triage_common.py](https://github.com/lin826/Local-SDFT/blob/main/scripts/triage_common.py); edit `POLICY` or the phase specs there, re-run `scripts/run_baselines.py` to re-export `data/inbox_triage.json`, and point `DATA_URL` at your fork.
2. **Noisier feedback** — flip a fraction of observed actions before training; watch the guardrail earn its keep.
3. **Widen the sweep** — try `K_SWEEP = (6, 12, 18, 24)`; does RAG ever stop serving stale weekday answers?
4. **Bigger model** — swap `MODEL_NAME` to `LiquidAI/LFM2.5-1.2B-Instruct`; 3-way routing is much steadier with more capacity.
5. **Full toolkit** — [Local-SDFT](https://github.com/lin826/Local-SDFT): the `sdft.online` serve-while-learning package (the real probe guardrail + auto-rollback), the `/data` live chat loop, and GRPO.

3-way on a 230M is at the edge — that's the fun. Curiosity, not VRAM, is the bottleneck.